# Sistemas de recomendación: Book-Crossing

**Objetivo.** Comparar recomendaciones basadas en popularidad, contenido y filtrado colaborativo, con énfasis en *fairness* entre grupos demográficos y de comportamiento.

Para realizar este trabajo nos basamos en lo que investigamos anteriormente pero en el proceso nos encontramos con que la forma en que lo habiamos pensado tal vez no era la adecuada. A continuacion dejamos el desarrollo del trabajo final donde aclaramos cada parte modificada del plan original.

## Indice
1. Descarga del dataset
2. Preprocesamiento y exploración
    
    2.a. Limpieza de valores nulos

    2.b Histogramas
3. Particionamiento de datos
4. Implementación de los algoritmos



## 1. Descarga del dataset

Se descargan las tres tablas del repositorio de Book-Crossing solo si todavía no existen localmente.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

base_url = 'https://raw.githubusercontent.com/ashwanidv100/Recommendation-System---Book-Crossing-Dataset/master/BX-CSV-Dump/'
files = ['BX-Book-Ratings.csv', 'BX-Books.csv', 'BX-Users.csv']

for file_name in files:
    destination = DATA_DIR / file_name
    if not destination.exists():
        print(f'Descargando {file_name}...')
        urlretrieve(base_url + file_name, destination)
    else:
        print(f'{file_name} ya existe')

BX-Book-Ratings.csv ya existe
BX-Books.csv ya existe
BX-Users.csv ya existe


In [2]:
read_options = dict(encoding='latin-1', sep=';', quotechar='"', on_bad_lines='skip', low_memory=False)
ratings = pd.read_csv(DATA_DIR / 'BX-Book-Ratings.csv', **read_options)
books = pd.read_csv(DATA_DIR / 'BX-Books.csv', **read_options)
users = pd.read_csv(DATA_DIR / 'BX-Users.csv', **read_options)

print('ratings:', ratings.shape, '| books:', books.shape, '| users:', users.shape)
display(ratings.head(), books.head(), users.head())

ratings: (1149780, 3) | books: (271360, 8) | users: (278858, 3)


,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


## 2. Preprocesamiento y exploración


### Limpieza de valores nulos

En Book-Crossing, un rating igual a `0` representa que el usuario no tuvo interaccion alguna, no cargo un raiting. Se conserva por separado, pero para entrenar y evaluar predicciones se usan ratings de 1 a 10. 

Las edades imposibles se consideran faltantes y los verdaderos valores faltantes no los imputamos para no generar falsa información, luego a la hora de agrupar por edad hicimos un grupo especial para los NaN.

In [3]:
def missing_report(df):
    return (df.isna().mean().mul(100)
              .rename('missing_%')
              .to_frame()
              .sort_values('missing_%', ascending=False))

print('Nulos en ratings')
display(missing_report(ratings))
print('Nulos en books')
display(missing_report(books))
print('Nulos en users')
display(missing_report(users))

# Normalización y deduplicación
ratings = ratings.drop_duplicates(['User-ID', 'ISBN'], keep='last').copy()
books = books.drop_duplicates('ISBN', keep='first').copy()
users = users.drop_duplicates('User-ID', keep='first').copy()

users['Age'] = pd.to_numeric(users['Age'], errors='coerce')
users.loc[~users['Age'].between(5, 100), 'Age'] = np.nan
users['Country'] = (users['Location'].fillna('unknown')
                     .str.rsplit(',', n=1).str[-1]
                     .str.strip().str.lower()
                     .replace('', 'unknown'))
books['Book-Title'] = books['Book-Title'].fillna('')
books['Book-Author'] = books['Book-Author'].fillna('Desconocido')

explicit_ratings = ratings.loc[ratings['Book-Rating'].between(1, 10)].copy()
print(f'Ratings explícitos: {len(explicit_ratings):,} de {len(ratings):,}')

Nulos en ratings


,missing_%
User-ID,0.0
ISBN,0.0
Book-Rating,0.0


Nulos en books


,missing_%
Image-URL-L,0.001106
Book-Author,0.000737
Publisher,0.000737
ISBN,0.000000
Book-Title,0.000000
Year-Of-Publication,0.000000
Image-URL-S,0.000000
Image-URL-M,0.000000


Nulos en users


,missing_%
Age,39.719857
User-ID,0.000000
Location,0.000000


Ratings explícitos: 433,671 de 1,149,780


### Histogramas

Graficamos por un lado los ratings reales (es decir de 1 a 10), y por otro la distribucion de las edades, utilizando 30 bins para ver su distribucion real y luego poder decidir como formar los grupos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=explicit_ratings, x='Book-Rating', ax=axes[0], color='steelblue')
axes[0].set(title='Distribución de ratings explícitos', xlabel='Rating', ylabel='Cantidad')
sns.histplot(users['Age'].dropna(), bins=20, ax=axes[1], color='darkorange')
axes[1].set(title='Distribución de edades válidas', xlabel='Edad', ylabel='Cantidad')
plt.tight_layout()
plt.show()

## 3. Particionamiento de los datos

Se realiza un *holdout* 70/30 por usuario. Se divide para cada usuario una parte de sus ratings para train y otra para test, de esta forma solo entran usuarios con al menos dos ratings explícitos para que ambos conjuntos tengan información. Este particionamiento se reutiliza en los modelos personalizados y evita evaluar con datos vistos durante el entrenamiento.

In [5]:
def split_by_user(df, train_fraction=0.70, seed=42):
    rng = np.random.default_rng(seed)
    train_indices, test_indices = [], []
    for _, group in df.groupby('User-ID'):
        if len(group) < 2:
            continue
        indices = group.index.to_numpy().copy()
        rng.shuffle(indices)
        cut = min(max(1, int(len(indices) * train_fraction)), len(indices) - 1)
        train_indices.extend(indices[:cut])
        test_indices.extend(indices[cut:])
    return df.loc[train_indices].copy(), df.loc[test_indices].copy()

train_ratings, test_ratings = split_by_user(explicit_ratings, seed=RANDOM_STATE)
print('Train:', train_ratings.shape, '| Test:', test_ratings.shape)
assert set(train_ratings.index).isdisjoint(test_ratings.index)

Train: (258302, 3) | Test: (129987, 3)


## 4. Implementación de los algoritmos

### 4.1 Modelos basados en popularidad

Primero se construye `data` con los nombres de columnas. Los promedios y conteos se calculan únicamente con ratings explícitos.

In [6]:
book_statistics = (explicit_ratings.groupby('ISBN')['Book-Rating']
                   .agg(average_rating='mean', ratings_count='count')
                   .reset_index())

data = (books[['ISBN', 'Book-Title', 'Book-Author']]
        .merge(book_statistics, on='ISBN', how='inner')
        .rename(columns={'Book-Title': 'title', 'Book-Author': 'authors'}))

data[['title', 'authors', 'average_rating', 'ratings_count']].head()

,title,authors,average_rating,ratings_count
0,Clara Callan,Richard Bruce Wright,7.666667,9
1,Decision in Normandy,Carlo D'Este,7.500000,2
2,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,7.833333,6
3,The Kitchen God's Wife,Amy Tan,8.176471,17
4,What If?: The World's Foremost Military Histor...,Robert Cowley,8.000000,1


#### Promedio de ratings

In [ ]:
# Ordeno de mayor a menor por promedio de ratings para luego recomendar los 10 primeros
top_rating = data.sort_values(
    by='average_rating',
    ascending=False
)

top_rating[['title', 'authors', 'average_rating', 'ratings_count']].head(10)

,title,authors,average_rating,ratings_count
39288,Fresh Milk: The Secret Life of Breasts,Fiona Giles,10.0,1
124506,Mysteries of Light: Meditations on the Mysteri...,"Pope John, II Paul",10.0,1
124509,Merry Hall (Beverley Nichols Trilogy Book 1),Beverley Nichols,10.0,1
28571,The Joy of Pickling: 200 Flavor-Packed Recipes...,Linda Ziedrich,10.0,1
124510,More Easy Classics to Moderns (Music for Milions),Music Sales Corporation,10.0,1
28566,The Oracle Doll,Catherine Dexter,10.0,1
124511,The Power to Heal,Francis MacNutt,10.0,1
28561,Intuitive Body: Aikido As a Clairsentient Prac...,Wendy Palmer,10.0,2
28560,Hagakure: The Book of the Samurai,Yamamoto Tsunetomo,10.0,1
124512,Hell's Angel. Mein Leben,Ralph Sonny Barger,10.0,1


#### Ranking con umbral de significancia

In [ ]:
min_votos = 100

top_populares = data[data['ratings_count'] >= min_votos]

top_populares = top_populares.sort_values(
    by='average_rating',
    ascending=False
)

top_populares[['title','authors','average_rating','ratings_count']].head(10)

: 

#### Fórmula IMDb

In [9]:
# Función dada en la práctica
def return_best_n(statistics, n, column):
    statistics = pd.DataFrame(statistics)
    return statistics.sort_values(by=column, axis=0, ascending=False).iloc[:n]

averages = data['average_rating']
counts = data['ratings_count']
minimum = data['ratings_count'].quantile(0.7)
general_average = data['average_rating'].mean()  # C

data['IMDb Metric'] = (data['average_rating'] * data['ratings_count'] + general_average * minimum) / (data['ratings_count'] + minimum)

best = return_best_n(data, 10, 'IMDb Metric')
best[['title', 'authors', 'average_rating', 'ratings_count', 'IMDb Metric']]

,title,authors,average_rating,ratings_count,IMDb Metric
56430,Harry Potter and the Chamber of Secrets Postca...,J. K. Rowling,9.869565,23,9.682203
5101,Postmarked Yesteryear: 30 Rare Holiday Postcards,Pamela E. Apkarian-Russell,10.000000,11,9.619622
47181,Dilbert: A Book of Postcards,Scott Adams,9.923077,13,9.603672
10818,"The Two Towers (The Lord of the Rings, Part 2)",J. R. R. Tolkien,9.720000,25,9.557596
11252,The Giving Tree,Shel Silverstein,9.750000,20,9.547958
54320,The Sneetches and Other Stories,Dr. Seuss,10.000000,8,9.505509
8568,Fox in Socks (I Can Read It All by Myself Begi...,Dr. Seuss,9.785714,14,9.503443
35395,Natural California: A Postcard Book,Not Applicable (Na ),10.000000,7,9.450565
17882,Uncle John's Supremely Satisfying Bathroom Rea...,Bathroom Readers Institute,10.000000,7,9.450565
12770,Calvin and Hobbes,Bill Watterson,9.583333,24,9.425196


### 4.2 Filtrado basado en contenido

Se combinan título y autor, se normaliza el texto, se remueven *stop-words*, se aplica *stemming* cuando NLTK está disponible y se vectoriza con `CountVectorizer`. Para evitar construir una matriz cuadrada enorme, la búsqueda usa vecinos más cercanos con distancia coseno sobre una muestra de libros con interacciones.

In [10]:
try:
    from nltk.stem.snowball import SnowballStemmer
    stemmer = SnowballStemmer('english')
    stem_word = stemmer.stem
except ImportError:
    print('NLTK no está instalado: se continúa sin stemming.')
    stem_word = lambda word: word

stop_words = set(ENGLISH_STOP_WORDS)

def text_analyzer(text):
    tokens = re.findall(r'[a-záéíóúüñ]+', str(text).lower())
    return [stem_word(token) for token in tokens if token not in stop_words and len(token) > 1]

# Límite ajustable para que la notebook pueda ejecutarse en una computadora personal.
MAX_CONTENT_BOOKS = 30_000
content_books = (data.sort_values('ratings_count', ascending=False)
                 .head(MAX_CONTENT_BOOKS)
                 .drop_duplicates('ISBN')
                 .reset_index(drop=True)
                 .copy())
content_books['text'] = content_books['title'] + ' ' + content_books['authors']

vectorizer = CountVectorizer(analyzer=text_analyzer, max_features=15_000)
content_matrix = vectorizer.fit_transform(content_books['text'])
content_model = NearestNeighbors(metric='cosine', algorithm='brute', n_jobs=-1)
content_model.fit(content_matrix)
print('Matriz de contenido:', content_matrix.shape)

Matriz de contenido: (30000, 15000)


In [11]:
def recommend_similar_books(title, n=10):
    matches = content_books.index[content_books['title'].str.contains(title, case=False, regex=False)]
    if len(matches) == 0:
        raise ValueError(f'No se encontró un libro que contenga: {title!r}')
    book_idx = matches[0]
    n_neighbors = min(n + 1, len(content_books))
    distances, indices = content_model.kneighbors(content_matrix[book_idx], n_neighbors=n_neighbors)
    result = content_books.iloc[indices[0]].copy()
    result['similarity'] = 1 - distances[0]
    return result.iloc[1:][['ISBN', 'title', 'authors', 'similarity']]

def recommend_content_for_user(user_id, n=10, positive_threshold=7):
    liked = train_ratings.loc[
        (train_ratings['User-ID'] == user_id) &
        (train_ratings['Book-Rating'] >= positive_threshold), 'ISBN'
    ]
    seed_indices = content_books.index[content_books['ISBN'].isin(liked)]
    if len(seed_indices) == 0:
        return pd.DataFrame(columns=['ISBN', 'title', 'authors', 'score'])
    query = np.asarray(content_matrix[seed_indices].mean(axis=0))
    distances, indices = content_model.kneighbors(query, n_neighbors=min(n + len(seed_indices), len(content_books)))
    seen = set(train_ratings.loc[train_ratings['User-ID'] == user_id, 'ISBN'])
    result = content_books.iloc[indices[0]].copy()
    result['score'] = 1 - distances[0]
    return result.loc[~result['ISBN'].isin(seen), ['ISBN', 'title', 'authors', 'score']].head(n)

# Ejemplo por título
recommend_similar_books('Harry Potter', n=10)

,ISBN,title,authors,similarity
102,0590353403,Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,0.824958
465,043936213X,Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,0.824958
26565,0439554896,Harry Potter and the Chamber of Secrets (Harry...,J. K. Rowling,0.783349
10374,043965548X,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling,0.783349
28769,0747561966,Harry Potter and the Philosopher's Stone,J.K. Rowling,0.774597
11702,0807281956,Harry Potter and the Sorcerer's Stone (Book 1 ...,J. K. Rowling,0.714435
18395,2070556859,Harry Potter et l'Ordre du PhÃ©nix (Harry Pott...,J.K. Rowling,0.694365
75,0439139597,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,0.645497
4579,8478884459,Harry Potter y la piedra filosofal,J. K. Rowling,0.645497
117,0439139600,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,0.645497


### 4.3 Definición de grupos y agregación de perfiles

Se crean grupos por edad, país e historial. Los países fuera de los más frecuentes se agrupan como `other`. Para la agregación *Least Misery*, el rating de un libro para el usuario virtual es el mínimo observado dentro de cada grupo. También se deja disponible el promedio para el análisis comparativo.

In [12]:
interaction_counts = explicit_ratings.groupby('User-ID').size().rename('interaction_count')
user_groups = users.merge(interaction_counts, on='User-ID', how='left')
user_groups['interaction_count'] = user_groups['interaction_count'].fillna(0).astype(int)

user_groups['age_group'] = pd.cut(
    user_groups['Age'], bins=[4, 17, 29, 44, 64, 100],
    labels=['05-17', '18-29', '30-44', '45-64', '65-100']
).astype('object').fillna('unknown')

top_countries = user_groups.loc[user_groups['Country'] != 'unknown', 'Country'].value_counts().head(10).index
user_groups['country_group'] = np.where(user_groups['Country'].isin(top_countries), user_groups['Country'], 'other')
active_counts = user_groups.loc[user_groups['interaction_count'] > 0, 'interaction_count']
history_threshold = active_counts.median()
user_groups['history_group'] = np.where(
    user_groups['interaction_count'] <= history_threshold, 'short', 'long'
)

print('Umbral de historial:', history_threshold)
display(user_groups[['User-ID', 'age_group', 'country_group', 'history_group', 'interaction_count']].head())

def aggregate_profiles(rating_data, group_column, strategy='least_misery'):
    if strategy not in {'least_misery', 'average'}:
        raise ValueError("strategy debe ser 'least_misery' o 'average'")
    merged = rating_data.merge(user_groups[['User-ID', group_column]], on='User-ID', how='inner')
    aggregation = 'min' if strategy == 'least_misery' else 'mean'
    return (merged.groupby([group_column, 'ISBN'], as_index=False)['Book-Rating']
            .agg(aggregation)
            .rename(columns={group_column: 'Virtual-User'}))

least_misery_age = aggregate_profiles(train_ratings, 'age_group', 'least_misery')
average_age = aggregate_profiles(train_ratings, 'age_group', 'average')
display(least_misery_age.head())

Umbral de historial: 1.0


,User-ID,age_group,country_group,history_group,interaction_count
0,1,unknown,usa,short,0
1,2,18-29,usa,short,0
2,3,unknown,other,short,0
3,4,05-17,portugal,short,0
4,5,unknown,united kingdom,short,0


,Virtual-User,ISBN,Book-Rating
0,05-17,0000000000,10
1,05-17,0002172755,8
2,05-17,0002246244,8
3,05-17,0004102169,4
4,05-17,0006167926,10


### 4.4 Filtrado colaborativo Usuario–Usuario

La implementación utiliza similitud coseno entre usuarios. Para mantener un consumo de memoria razonable se trabaja con los usuarios y libros más activos; estos límites son parámetros explícitos y pueden ampliarse según el equipo disponible.

In [13]:
MAX_USERS = 5_000
MAX_BOOKS = 5_000
N_NEIGHBORS = 30

active_users = train_ratings['User-ID'].value_counts().head(MAX_USERS).index
active_books = train_ratings.loc[train_ratings['User-ID'].isin(active_users), 'ISBN'].value_counts().head(MAX_BOOKS).index
collab_train = train_ratings[
    train_ratings['User-ID'].isin(active_users) & train_ratings['ISBN'].isin(active_books)
].copy()

user_ids = pd.Index(collab_train['User-ID'].unique(), name='User-ID')
book_ids = pd.Index(collab_train['ISBN'].unique(), name='ISBN')
user_to_idx = pd.Series(np.arange(len(user_ids)), index=user_ids)
book_to_idx = pd.Series(np.arange(len(book_ids)), index=book_ids)
rows = collab_train['User-ID'].map(user_to_idx).to_numpy()
cols = collab_train['ISBN'].map(book_to_idx).to_numpy()
values = collab_train['Book-Rating'].astype(float).to_numpy()
user_item = csr_matrix((values, (rows, cols)), shape=(len(user_ids), len(book_ids)))

user_means = np.asarray(user_item.sum(axis=1)).ravel() / np.maximum(user_item.getnnz(axis=1), 1)
centered_values = values - user_means[rows]
centered_matrix = csr_matrix((centered_values, (rows, cols)), shape=user_item.shape)

user_model = NearestNeighbors(metric='cosine', algorithm='brute', n_jobs=-1)
user_model.fit(centered_matrix)
k = min(N_NEIGHBORS + 1, len(user_ids))
neighbor_distances, neighbor_indices = user_model.kneighbors(centered_matrix, n_neighbors=k)
neighbor_similarities = 1 - neighbor_distances
print('Matriz usuario–libro:', user_item.shape, '| ratings:', user_item.nnz)

Matriz usuario–libro: (4619, 5000) | ratings: 55414


In [14]:
def predict_user_user(user_id, isbn):
    if user_id not in user_to_idx.index or isbn not in book_to_idx.index:
        return float(explicit_ratings['Book-Rating'].mean())
    u = int(user_to_idx[user_id])
    i = int(book_to_idx[isbn])
    neighbors = neighbor_indices[u, 1:]
    similarities = neighbor_similarities[u, 1:]
    neighbor_ratings = user_item[neighbors, i].toarray().ravel()
    rated = neighbor_ratings > 0
    if not rated.any() or np.abs(similarities[rated]).sum() == 0:
        return float(user_means[u])
    deviations = neighbor_ratings[rated] - user_means[neighbors[rated]]
    prediction = user_means[u] + np.dot(similarities[rated], deviations) / np.abs(similarities[rated]).sum()
    return float(np.clip(prediction, 1, 10))

def recommend_user_user(user_id, n=10):
    if user_id not in user_to_idx.index:
        return pd.DataFrame(columns=['ISBN', 'title', 'authors', 'prediction'])
    u = int(user_to_idx[user_id])
    seen = set(collab_train.loc[collab_train['User-ID'] == user_id, 'ISBN'])
    candidates = [isbn for isbn in book_ids if isbn not in seen]
    scores = np.array([predict_user_user(user_id, isbn) for isbn in candidates])
    top = np.argsort(scores)[-n:][::-1]
    result = pd.DataFrame({'ISBN': np.array(candidates)[top], 'prediction': scores[top]})
    return result.merge(data[['ISBN', 'title', 'authors']], on='ISBN', how='left')[['ISBN', 'title', 'authors', 'prediction']]

example_user = collab_train['User-ID'].value_counts().index[0]
recommend_user_user(example_user, n=10)

,ISBN,title,authors,prediction
0,0060508302,Angels Everywhere: A Season of Angels/Touched ...,Debbie MacOmber,10.000000
1,0312050631,Confessions of a Failed Southern Lady,Florence King,10.000000
2,0671617028,The Color Purple,Alice Walker,10.000000
3,0140097058,In the Country of Last Things (Contemporary Am...,Paul Auster,9.969482
4,0553260960,"The Mammoth Hunters (Auel, Jean M. , Earth's C...",Jean M. Auel,9.912172
5,0451203771,Scarlet Feather,Maeve Binchy,9.789995
6,0553280414,A Separate Peace,John Knowles,9.727057
7,0553274295,Where the Red Fern Grows,Wilson Rawls,9.727057
8,0671741195,The Cradle Will Fall,Mary Higgins Clark,9.713071
9,0375725784,A Heartbreaking Work of Staggering Genius,Dave Eggers,9.636148


## 5. Evaluación y análisis

### 5.1 MAE del filtrado colaborativo

El MAE se calcula globalmente, por usuario y por grupo. Se limita la cantidad de filas de evaluación para que la primera ejecución sea rápida; para el informe final puede aumentarse `MAX_EVAL_RATINGS` o asignarle `None`.

In [15]:
MAX_EVAL_RATINGS = 20_000
evaluable_test = test_ratings[
    test_ratings['User-ID'].isin(user_ids) & test_ratings['ISBN'].isin(book_ids)
].copy()
if MAX_EVAL_RATINGS is not None and len(evaluable_test) > MAX_EVAL_RATINGS:
    evaluable_test = evaluable_test.sample(MAX_EVAL_RATINGS, random_state=RANDOM_STATE)

evaluable_test['prediction'] = [
    predict_user_user(user, isbn)
    for user, isbn in zip(evaluable_test['User-ID'], evaluable_test['ISBN'])
]
evaluable_test['absolute_error'] = (
    evaluable_test['Book-Rating'] - evaluable_test['prediction']
).abs()

mae_global = evaluable_test['absolute_error'].mean()
mae_by_user = evaluable_test.groupby('User-ID', as_index=False)['absolute_error'].mean().rename(
    columns={'absolute_error': 'MAE'}
)
mae_by_user = mae_by_user.merge(
    user_groups[['User-ID', 'age_group', 'country_group', 'history_group']], on='User-ID', how='left'
)
print(f'MAE global: {mae_global:.4f}')
display(mae_by_user.head())

for group_col in ['age_group', 'country_group', 'history_group']:
    display(mae_by_user.groupby(group_col)['MAE'].agg(['mean', 'std', 'count']).sort_values('mean'))

MAE global: 1.2541


,User-ID,MAE,age_group,country_group,history_group
0,243,1.058930,unknown,usa,long
1,254,0.978020,18-29,usa,long
2,388,1.000000,30-44,usa,long
3,503,0.000000,30-44,usa,long
4,507,1.047278,unknown,usa,long


,mean,std,count
age_group,,,
65-100,1.043275,0.886576,54
05-17,1.247135,1.025074,89
45-64,1.277769,0.870720,565
30-44,1.290721,0.831125,1294
18-29,1.298293,0.812672,967
unknown,1.427063,0.857438,999


,mean,std,count
country_group,,,
new zealand,1.127237,0.962160,18
australia,1.172048,0.748009,74
spain,1.233880,0.923312,41
canada,1.264637,0.745429,388
france,1.288351,0.868264,40
germany,1.290372,1.048774,85
other,1.310123,0.843006,311
usa,1.332016,0.840551,2782
italy,1.335573,1.468082,12


,mean,std,count
history_group,,,
long,1.320703,0.846891,3968


### 5.2 Métricas de calidad

- **Acierto@K:** proporción de usuarios para los que al menos un libro recomendado aparece entre sus ratings positivos ocultos de test.
- **Diversidad:** proporción de pares de libros recomendados que tienen autores diferentes.
- **Serendipia:** proporción de recomendaciones relevantes y fuera del 30% de libros más populares.
- **Cobertura:** proporción del catálogo que aparece al menos una vez en las listas.

In [16]:
isbn_to_author = books.set_index('ISBN')['Book-Author'].to_dict()
popularity_cutoff = data['ratings_count'].quantile(0.70)
popular_isbns = set(data.loc[data['ratings_count'] >= popularity_cutoff, 'ISBN'])

def diversity_author(recommended_isbns):
    authors = [isbn_to_author.get(isbn, 'Desconocido') for isbn in recommended_isbns]
    if len(authors) < 2:
        return np.nan
    different, total = 0, 0
    for i in range(len(authors)):
        for j in range(i + 1, len(authors)):
            different += authors[i] != authors[j]
            total += 1
    return different / total

def evaluate_recommendation_lists(recommendations, test_data, catalog, relevant_threshold=7):
    relevant = (test_data[test_data['Book-Rating'] >= relevant_threshold]
                .groupby('User-ID')['ISBN'].apply(set).to_dict())
    hits, diversities, serendipities, all_recommended = [], [], [], set()
    for user_id, recs in recommendations.items():
        recs = list(recs)
        relevant_items = relevant.get(user_id, set())
        hits.append(int(bool(set(recs) & relevant_items)))
        diversities.append(diversity_author(recs))
        serendipitous_hits = {isbn for isbn in recs if isbn in relevant_items and isbn not in popular_isbns}
        serendipities.append(len(serendipitous_hits) / max(len(recs), 1))
        all_recommended.update(recs)
    return pd.Series({
        'hit_rate@k': np.mean(hits),
        'diversity_author': np.nanmean(diversities),
        'serendipity@k': np.mean(serendipities),
        'coverage': len(all_recommended) / max(len(catalog), 1),
        'evaluated_users': len(recommendations),
    })

EVAL_USERS = 100
evaluation_users = list(pd.Index(evaluable_test['User-ID'].unique())[:EVAL_USERS])
K = 10

popularity_list = best['ISBN'].head(K).tolist()
popularity_recs = {user_id: popularity_list for user_id in evaluation_users}
content_recs = {
    user_id: recommend_content_for_user(user_id, K)['ISBN'].tolist()
    for user_id in evaluation_users
}
collaborative_recs = {
    user_id: recommend_user_user(user_id, K)['ISBN'].tolist()
    for user_id in evaluation_users
}

quality_results = pd.DataFrame({
    'popularidad_IMDb': evaluate_recommendation_lists(popularity_recs, test_ratings, data['ISBN']),
    'contenido': evaluate_recommendation_lists(content_recs, test_ratings, content_books['ISBN']),
    'colaborativo': evaluate_recommendation_lists(collaborative_recs, test_ratings, book_ids),
}).T
quality_results

,hit_rate@k,diversity_author,serendipity@k,coverage,evaluated_users
popularidad_IMDb,0.01,0.977778,0.0,0.000067,100.0
contenido,0.25,0.543556,0.0,0.026467,100.0
colaborativo,0.05,0.979333,0.0,0.150600,100.0


### 5.3 Análisis estadístico y pruebas de hipótesis

El t-test independiente compara dos grupos diferentes. ANOVA compara tres o más grupos. Para comparar dos modelos evaluados sobre los mismos usuarios se debe usar un t-test pareado. Antes del informe final conviene verificar normalidad y homogeneidad de varianzas; si no se cumplen, usar Mann–Whitney, Kruskal–Wallis o Wilcoxon.

In [17]:
def independent_ttest(metric_df, group_column, group_a, group_b, metric='MAE'):
    a = metric_df.loc[metric_df[group_column] == group_a, metric].dropna()
    b = metric_df.loc[metric_df[group_column] == group_b, metric].dropna()
    statistic, pvalue = stats.ttest_ind(a, b, equal_var=False)
    return pd.Series({'group_a': group_a, 'group_b': group_b, 'statistic': statistic, 'p_value': pvalue})

def one_way_anova(metric_df, group_column, metric='MAE', min_group_size=2):
    samples = [group[metric].dropna().to_numpy() for _, group in metric_df.groupby(group_column)]
    samples = [sample for sample in samples if len(sample) >= min_group_size]
    if len(samples) < 2:
        return pd.Series({'statistic': np.nan, 'p_value': np.nan, 'n_groups': len(samples)})
    statistic, pvalue = stats.f_oneway(*samples)
    return pd.Series({'statistic': statistic, 'p_value': pvalue, 'n_groups': len(samples)})

def paired_model_ttest(model_a_by_user, model_b_by_user, metric='MAE'):
    paired = model_a_by_user[['User-ID', metric]].merge(
        model_b_by_user[['User-ID', metric]], on='User-ID', suffixes=('_a', '_b')
    ).dropna()
    statistic, pvalue = stats.ttest_rel(paired[f'{metric}_a'], paired[f'{metric}_b'])
    return pd.Series({'statistic': statistic, 'p_value': pvalue, 'paired_users': len(paired)})

print('T-test, historial corto vs. largo')
display(independent_ttest(mae_by_user, 'history_group', 'short', 'long'))
print('ANOVA por edad')
display(one_way_anova(mae_by_user[mae_by_user['age_group'] != 'unknown'], 'age_group'))
print('ANOVA por país')
display(one_way_anova(mae_by_user, 'country_group'))

T-test, historial corto vs. largo


group_a      short
group_b       long
statistic      NaN
p_value        NaN
dtype: object

ANOVA por edad


statistic    1.248276
p_value      0.288265
n_groups     5.000000
dtype: float64

ANOVA por país


statistic     0.817579
p_value       0.611689
n_groups     11.000000
dtype: float64

### 5.4 Fairness

Una diferencia estadísticamente significativa no indica por sí sola la magnitud del sesgo. Por eso se reportan también la brecha absoluta y la razón entre el mejor y el peor MAE medio de cada segmentación.

In [18]:
def fairness_summary(metric_df, group_column, metric='MAE', min_users=5):
    summary = metric_df.groupby(group_column)[metric].agg(['mean', 'std', 'count'])
    summary = summary[summary['count'] >= min_users].sort_values('mean')
    if summary.empty:
        return summary, pd.Series(dtype=float)
    disparity = pd.Series({
        'absolute_gap': summary['mean'].max() - summary['mean'].min(),
        'worst_to_best_ratio': summary['mean'].max() / summary['mean'].min(),
        'best_group': summary['mean'].idxmin(),
        'worst_group': summary['mean'].idxmax(),
    })
    return summary, disparity

for criterion in ['age_group', 'country_group', 'history_group']:
    group_summary, disparity = fairness_summary(mae_by_user, criterion)
    print(f'Fairness según {criterion}')
    display(group_summary, disparity.to_frame('value'))

Fairness según age_group


,mean,std,count
age_group,,,
65-100,1.043275,0.886576,54
05-17,1.247135,1.025074,89
45-64,1.277769,0.870720,565
30-44,1.290721,0.831125,1294
18-29,1.298293,0.812672,967
unknown,1.427063,0.857438,999


,value
absolute_gap,0.383788
worst_to_best_ratio,1.367868
best_group,65-100
worst_group,unknown


Fairness según country_group


,mean,std,count
country_group,,,
new zealand,1.127237,0.962160,18
australia,1.172048,0.748009,74
spain,1.233880,0.923312,41
canada,1.264637,0.745429,388
france,1.288351,0.868264,40
germany,1.290372,1.048774,85
other,1.310123,0.843006,311
usa,1.332016,0.840551,2782
italy,1.335573,1.468082,12


,value
absolute_gap,0.349206
worst_to_best_ratio,1.309789
best_group,new zealand
worst_group,portugal


Fairness según history_group


,mean,std,count
history_group,,,
long,1.320703,0.846891,3968


,value
absolute_gap,0.0
worst_to_best_ratio,1.0
best_group,long
worst_group,long


## 6. Extensiones experimentales

Las siguientes configuraciones dejan preparadas las tres modificaciones propuestas en el informe:

1. **Validación cruzada:** repetir el entrenamiento y la evaluación con varias semillas y consolidar media y desvío del MAE.
2. **Agregación por promedio:** comparar `average_age` con `least_misery_age` usando exactamente el mismo conjunto de libros y grupos.
3. **Mayor granularidad:** reemplazar los cortes etarios base por una lista más fina y volver a ejecutar agrupación, entrenamiento y evaluación.

Es importante reentrenar el modelo dentro de cada fold; no se deben calcular perfiles ni similitudes usando datos del fold de test.

Nosotras habiamos pensado hacer 3 modelos y entrenar cada uno con los perfiles de usuarios virtuales segun cada agrupamiento y despues nos dimos cuenta que no tenia sentido porque terminamos teninedo muy pocos usuarios, esta tecnica sirve para cuando tenes muchos grupos, ponele paises (hubiera tenido mas sentido). Terminamos entrenando un solo modelo con los datos reales, y luego agrupar de diferentes amneras para comparar si hay un grupo de personas quedan con una mala prediccion, entonces perjudicamos a un grupo en especifico.
Por lo tanto no usamos lo de least_misey y average porque eos es para la agregacion de perfiles 

In [19]:
# Configuración base para repetir el experimento completo.
CV_SEEDS = [42, 123, 2026, 7, 99]
cv_splits = [split_by_user(explicit_ratings, train_fraction=0.80, seed=seed) for seed in CV_SEEDS]
print(f'{len(cv_splits)} particiones preparadas; reentrenar y evaluar el modelo en cada una.')

# Comparación descriptiva de estrategias de agregación para los grupos etarios.
aggregation_comparison = (least_misery_age.rename(columns={'Book-Rating': 'least_misery'})
                          .merge(average_age.rename(columns={'Book-Rating': 'average'}),
                                 on=['Virtual-User', 'ISBN'], how='inner'))
aggregation_comparison['difference'] = aggregation_comparison['average'] - aggregation_comparison['least_misery']
display(aggregation_comparison.groupby('Virtual-User')[['least_misery', 'average', 'difference']].mean())

# Segmentación etaria alternativa, más granular.
fine_age_groups = pd.cut(
    user_groups['Age'],
    bins=[4, 12, 17, 24, 34, 44, 54, 64, 74, 100],
    labels=['05-12', '13-17', '18-24', '25-34', '35-44', '45-54', '55-64', '65-74', '75-100']
).astype('object').fillna('unknown')
user_groups['age_group_fine'] = fine_age_groups
least_misery_age_fine = aggregate_profiles(train_ratings, 'age_group_fine', 'least_misery')
least_misery_age_fine.head()

5 particiones preparadas; reentrenar y evaluar el modelo en cada una.


,least_misery,average,difference
Virtual-User,,,
05-17,7.607393,7.721104,0.113712
18-29,7.394831,7.647078,0.252247
30-44,7.292827,7.575166,0.282339
45-64,7.682785,7.881487,0.198702
65-100,7.593709,7.656771,0.063062
unknown,6.971364,7.255479,0.284115


,Virtual-User,ISBN,Book-Rating
0,05-12,0006512046,9
1,05-12,0006751504,10
2,05-12,002542730X,7
3,05-12,0027888355,10
4,05-12,0060096195,10
